In [1]:
import triton
import triton.language as tl
import torch 

In [3]:
@triton.jit
def Matmul_kernel(a_ptr,b_ptr,c_ptr,
                  M,K,N,
                  stride_am,stride_ak,
                  stride_bk,stride_bn,
                  stride_cm,stride_cn,
                  BLOCK_SIZE_M:tl.constexpr,
                  BLOCK_SIZE_N:tl.constexpr,
                  BLOCK_SIZE_K:tl.constexpr
                 ):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offsets_m = pid_m*BLOCK_SIZE_M + tl.arange(0,BLOCK_SIZE_M)
    offsets_n = pid_n*BLOCK_SIZE_N + tl.arange(0,BLOCK_SIZE_N)
    maskM = offsets_m < M
    maskN = offsets_n < N
    acc = tl.zeros((BLOCK_SIZE_M,BLOCK_SIZE_N),dtype = tl.float32)
    # for k in tl.arange(0,tl.cdiv(K,BLOCK_SIZE_K),)
    for k in range(0,K,BLOCK_SIZE_K):
        offsets_k = k + tl.arange(0,BLOCK_SIZE_K)
        maskK = offsets_k < K
        A_ptrs = a_ptr + offsets_m[:,None] * stride_am + offsets_k[None,:] * stride_ak
        B_ptrs = b_ptr + offsets_k[:,None] * stride_bk + offsets_n[None,:] * stride_bn
        mask_a = maskM[:,None] & maskK[None,:]
        mask_b = maskK[:,None] & maskN[None,:]
        a = tl.load(A_ptrs,mask=mask_a,other = 0.0)
        b = tl.load(B_ptrs,mask=mask_b,other = 0.0)
        acc = tl.dot(a,b,acc)
        
    C_ptrs = c_ptr+offsets_m[:,None] * stride_cm + offsets_n[None,:] * stride_cn
    mask_c = maskM[:,None] & maskN[None,:]
    tl.store(C_ptrs,acc,mask=mask_c)
    
def runkernel(a,b,c):
    BLOCK_SIZE_M = 32
    BLOCK_SIZE_N = 32
    BLOCK_SIZE_K = 32 
    Matmul_kernel[(triton.cdiv(1024,BLOCK_SIZE_M),
                  triton.cdiv(1536,BLOCK_SIZE_N))](
        a,b,c,
        1024,768,1536,
        a.stride(0), a.stride(1),  # stride_am, stride_ak
        b.stride(0), b.stride(1),  # stride_bk, stride_bn
        c.stride(0), c.stride(1),  # stride_cm, stride_cn
        BLOCK_SIZE_M,BLOCK_SIZE_N,BLOCK_SIZE_K
                  )

In [4]:
@triton.autotune(
    configs=[triton.Config(
            {
                'BLOCK_SIZE_M': BM, 'BLOCK_SIZE_N': BN, "BLOCK_SIZE_K": BK, "GROUP_SIZE_M": GM
            })
        for BM in [32,64,128]  #
        for BN in [32,64,128]  #
        for BK in [32,64,128] #
        for GM in [2,4,8]],
    key=["M", "K", "N"], # 作为调优的标识符 
)
@triton.jit
def Matmul_kernel_v2(a_ptr,b_ptr,c_ptr,
                  M,K,N,
                  stride_am,stride_ak,
                  stride_bk,stride_bn,
                  stride_cm,stride_cn,
                  BLOCK_SIZE_M:tl.constexpr,
                  BLOCK_SIZE_N:tl.constexpr,
                  BLOCK_SIZE_K:tl.constexpr
                 ):
    
    pid = tl.program_id(axis = 0)
    num_pid_m = tl.cdiv(M,BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N,BLOCK_SIZE_N)
    num_block_per_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_block_per_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(GROUP_SIZE_M,num_pid_m-first_pid_m)
    pid_m = (pid % num_block_per_group) % group_size_m + first_pid_m
    pid_n = (pid % num_block_per_group) // group_size_m

    offsets_m = pid_m*BLOCK_SIZE_M + tl.arange(0,BLOCK_SIZE_M)
    offsets_n = pid_n*BLOCK_SIZE_N + tl.arange(0,BLOCK_SIZE_N)
    maskM = offsets_m < M
    maskN = offsets_n < N
    acc = tl.zeros((BLOCK_SIZE_M,BLOCK_SIZE_N),dtype = tl.float32)
    # for k in tl.arange(0,tl.cdiv(K,BLOCK_SIZE_K),)
    for k in range(0,K,BLOCK_SIZE_K):
        offsets_k = k + tl.arange(0,BLOCK_SIZE_K)
        maskK = offsets_k < K
        A_ptrs = a_ptr + offsets_m[:,None] * stride_am + offsets_k[None,:] * stride_ak
        B_ptrs = b_ptr + offsets_k[:,None] * stride_bk + offsets_n[None,:] * stride_bn
        mask_a = maskM[:,None] & maskK[None,:]
        mask_b = maskK[:,None] & maskN[None,:]
        a = tl.load(A_ptrs,mask=mask_a,other = 0.0)
        b = tl.load(B_ptrs,mask=mask_b,other = 0.0)
        acc = tl.dot(a,b,acc)
        
    C_ptrs = c_ptr+offsets_m[:,None] * stride_cm + offsets_n[None,:] * stride_cn
    mask_c = maskM[:,None] & maskN[None,:]
    tl.store(C_ptrs,acc,mask=mask_c)

In [8]:
@triton.autotune(
    configs=[triton.Config(
            {
                'BLOCK_SIZE_M': BM, 'BLOCK_SIZE_N': BN, "BLOCK_SIZE_K": BK, "GROUP_SIZE_M": GM
            },num_stages=2, num_warps=4)
        for BM in [32,64,128]  #
        for BN in [32,64,128]  #
        for BK in [32,64,128] #
        for GM in [2,4,8]],
    key=["M", "K", "N"], # 作为调优的标识符 
)
@triton.jit
def Matmul_kernel_V3(a_ptr,b_ptr,c_ptr,
                  M,K,N,
                  stride_am,stride_ak,
                  stride_bk,stride_bn,
                  stride_cm,stride_cn,    
                  BLOCK_SIZE_M: tl.constexpr,  #
                  BLOCK_SIZE_N: tl.constexpr,  #
                  BLOCK_SIZE_K: tl.constexpr,  #
                  GROUP_SIZE_M: tl.constexpr,  #
                  NUM_SMS: tl.constexpr,  #
                 ):
    pid = tl.program_id(axis = 0)

    
    nums_block_m = tl.cdiv(M,BLOCK_SIZE_M)
    nums_block_n = tl.cdiv(N,BLOCK_SIZE_N)
    nums_block = nums_block_m*nums_block_n
    
    
    # pid_m,pid_n = ComputePid(pid,M,N,BLOCK_SIZE_M,BLOCK_SIZE_N,GROUP_SIZE_M)
    # offsets_m = pid_m*BLOCK_SIZE_M + tl.arange(0,BLOCK_SIZE_M)
    # offsets_n = pid_n*BLOCK_SIZE_N + tl.arange(0,BLOCK_SIZE_N)
    # maskM = offsets_m < M
    # maskN = offsets_n < N
    # acc = tl.zeros((BLOCK_SIZE_M,BLOCK_SIZE_N),dtype = tl.float32)

    for start_pid in tl.range(pid,nums_block,NUM_SMS,flatten=True):
        
        pid_m,pid_n = _compute_pid(start_pid,nums_block,nums_block_m,GROUP_SIZE_M)
        offs_m = pid_m*BLOCK_SIZE_M + tl.arange(0,BLOCK_SIZE_M)
        offs_n = pid_n*BLOCK_SIZE_N + tl.arange(0,BLOCK_SIZE_N)

        acc = tl.zeros((BLOCK_SIZE_M,BLOCK_SIZE_N),dtype = tl.float32)
        
        k_tiles = K // BLOCK_SIZE_K       # 完整 tile 数
        k_remainder = K % BLOCK_SIZE_K    # 剩余元素数
    
        # ========== 主循环：无 mask，全速向量化加载 ==========
        for ki in range(0, k_tiles):
            k = ki * BLOCK_SIZE_K
            offs_k = k + tl.arange(0, BLOCK_SIZE_K)
            # 无需 maskK！offs_m < M 仍需保留（M 不被 BK 整除）
            a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
            b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn
            mask_a = (offs_m[:, None] < M)              
            mask_b = (offs_n[None, :] < N)              
            a = tl.load(a_ptrs, mask=mask_a, other=0.0)
            b = tl.load(b_ptrs, mask=mask_b, other=0.0)
            acc = tl.dot(a, b, acc)
    
        # ========== 尾部处理：仅在有余数时执行 ==========
        if k_remainder > 0:  # ✅ 编译期可求值时直接消除整个分支
            k = k_tiles * BLOCK_SIZE_K
            offs_k = k + tl.arange(0, BLOCK_SIZE_K)
            mask_k = offs_k < K                          # 仅此一处需要 K-mask
            a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
            b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn
            mask_a = (offs_m[:, None] < M) & mask_k[None, :]
            mask_b = mask_k[:, None] & (offs_n[None, :] < N)
            a = tl.load(a_ptrs, mask=mask_a, other=0.0)
            b = tl.load(b_ptrs, mask=mask_b, other=0.0)
            acc = tl.dot(a, b, acc)
        
        C_ptrs = c_ptr+offs_m[:,None] * stride_cm + offs_n[None,:] * stride_cn
        mask_c = (offs_m[:,None] < M) & (offs_n[None,:] < N)
        tl.store(C_ptrs,acc,mask=mask_c)
    
def runkernel_V3(a,b,c):
    NUM_SMS = torch.cuda.get_device_properties("cuda").multi_processor_count
    # grid = lambda meta:(triton.cdiv(1024,meta["BLOCK_SIZE_M"]),triton.cdiv(1536,meta["BLOCK_SIZE_N"]))
    grid = lambda meta: (min(NUM_SMS, 
                         triton.cdiv(1024, meta["BLOCK_SIZE_M"]) * 
                         triton.cdiv(1536, meta["BLOCK_SIZE_N"])), )
    
    Matmul_kernel_V3[grid](
            a,b,c,
            1024,768,1536,
            a.stride(0), a.stride(1),  # stride_am, stride_ak
            b.stride(0), b.stride(1),  # stride_bk, stride_bn
            c.stride(0), c.stride(1),  # stride_cm, stride_cn
            NUM_SMS = NUM_SMS,
                      )
